> We *used* these vectors to **find related words** — but they were never trained for similarity. They were trained on a **prediction task**, and similarity fell out as a happy side effect.

### Two flavours of Word2Vec

A word is defined by the company it keeps. We can turn that into a prediction task in two symmetric ways:

- **CBOW** (Continuous Bag of Words): given the **context**, predict the **missing word** — $P(w_i \mid w_{i-2}, w_{i-1}, w_{i+1}, w_{i+2})$
- **Skip-gram**: flip it — given the **word**, predict its **context** — $P(w_{i-2}, w_{i-1}, w_{i+1}, w_{i+2} \mid w_i)$

Two sides of the same coin. **In this notebook we build CBOW**: predict a word from the words around it.

# Data

## 1. Get a corpus

Original model was trained on a 100 billion words part of Google News Corpus. 
I don't think I can find it, and we are for sure not going to be able to use it.

We'll stick with Wikitext 2 version 1

Links: [Wikitext 2 Description](https://paperswithcode.com/dataset/wikitext-2) [Wikitext 2 Datasets Page](https://huggingface.co/datasets/wikitext/viewer/wikitext-2-v1/train)

In [24]:
from collections import Counter 
from tqdm.auto import tqdm, trange
from datasets import load_dataset
from matplotlib import pyplot as plt
import numpy as np
from pathlib import Path
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader

In [25]:
# Load the dataset
wikitext = load_dataset("wikitext", "wikitext-2-v1")
wikitext, wikitext['train'][10]['text']

(DatasetDict({
     test: Dataset({
         features: ['text'],
         num_rows: 4358
     })
     train: Dataset({
         features: ['text'],
         num_rows: 36718
     })
     validation: Dataset({
         features: ['text'],
         num_rows: 3760
     })
 }),
 ' The game \'s battle system , the <unk> system , is carried over directly from <unk> Chronicles . During missions , players select each unit using a top @-@ down perspective of the battlefield map : once a character is selected , the player moves the character around the battlefield in third @-@ person . A character can only act once per @-@ turn , but characters can be granted multiple turns at the expense of other characters \' turns . Each character has a field and distance of movement limited by their Action <unk> . Up to nine characters can be assigned to a single mission . During gameplay , characters will call out if something happens to them , such as their health points ( HP ) getting low or being knocked 

In [ ]:
# Discrimnative
# p(y_costofapartment= 200| x_sizeofapartment=42)

# Generative
# p(x_sizeofapartment=42) = 

# How though?
# p(i am going to make him an offer he cant refuse) ]

# p(offer he cant refuse)
# Language Models - GPT, Bert, Transfomers... - Autoregressive Language Models; Next Word Prediction Task..
# = p(offer) * p(he | offer) * p(cant | offer he) * p(refuse | offer he cant)

# make him an offer he cant refuse
# 1. make him offer he -> an
# 2. him an he cant -> offer
# 3. an offer cant refuse -> he
# P(he | an offer cant refuse)

# Goals

PS: we do not replicate the paper Word2Vec but do something in the same spirit for now

**predict the word given its surrounding words**

![Screenshot 2025-05-05 at 22.05.25.png](<attachment:Screenshot 2025-05-05 at 22.05.25.png>)


This is what we try to model: $ P(w_i | w_{i-2}, w_{i-1}, w_{i+1} \dots, w_{i+2})$ 

e.g. $P({\text{he} | \text{an},\text{offer}, \text{can't}, \text{refuse}})$

# Lets start by pre-processing the data

## Get a Vocab

You know the drill by now. Get a counter. Set n_words. Select top-n.

In [34]:
 wikitext['train'][10]['text'].split()

['The',
 'game',
 "'s",
 'battle',
 'system',
 ',',
 'the',
 '<unk>',
 'system',
 ',',
 'is',
 'carried',
 'over',
 'directly',
 'from',
 '<unk>',
 'Chronicles',
 '.',
 'During',
 'missions',
 ',',
 'players',
 'select',
 'each',
 'unit',
 'using',
 'a',
 'top',
 '@-@',
 'down',
 'perspective',
 'of',
 'the',
 'battlefield',
 'map',
 ':',
 'once',
 'a',
 'character',
 'is',
 'selected',
 ',',
 'the',
 'player',
 'moves',
 'the',
 'character',
 'around',
 'the',
 'battlefield',
 'in',
 'third',
 '@-@',
 'person',
 '.',
 'A',
 'character',
 'can',
 'only',
 'act',
 'once',
 'per',
 '@-@',
 'turn',
 ',',
 'but',
 'characters',
 'can',
 'be',
 'granted',
 'multiple',
 'turns',
 'at',
 'the',
 'expense',
 'of',
 'other',
 'characters',
 "'",
 'turns',
 '.',
 'Each',
 'character',
 'has',
 'a',
 'field',
 'and',
 'distance',
 'of',
 'movement',
 'limited',
 'by',
 'their',
 'Action',
 '<unk>',
 '.',
 'Up',
 'to',
 'nine',
 'characters',
 'can',
 'be',
 'assigned',
 'to',
 'a',
 'single',
 'm

In [29]:
c = Counter()
c.update(['a','b', 'c', 'c', 'c','a'])

c

Counter({'c': 3, 'a': 2, 'b': 1})

In [35]:
# Make a vocab
word_counter = Counter()
n_words = 10_000

for doc in wikitext['train']:
    word_counter.update(doc['text'].split())
    
len(word_counter), word_counter.most_common(10)

(33277,
 [('the', 113161),
  (',', 99913),
  ('.', 73388),
  ('of', 56889),
  ('<unk>', 54625),
  ('and', 50603),
  ('in', 39453),
  ('to', 39190),
  ('a', 34237),
  ('=', 29570)])

In [36]:
# Use the word frequencies to make the vocab
unk_token = '<unk>'
vocab = {'<unk>': 0, '<pad>': 1}
for word, freq in word_counter.most_common(n_words):
    if word not in vocab:
        vocab[word] = len(vocab)

print(vocab['however'])

347


In [45]:
x = []
for token in wikitext['train'][10]['text'].split():
    x.append(vocab.get(token, vocab['<unk>']))

x

[13,
 77,
 17,
 718,
 258,
 3,
 2,
 0,
 258,
 3,
 24,
 1090,
 71,
 1176,
 25,
 0,
 4419,
 4,
 216,
 2668,
 3,
 467,
 5488,
 174,
 1238,
 437,
 9,
 309,
 14,
 197,
 3911,
 5,
 2,
 5750,
 3910,
 43,
 511,
 9,
 264,
 24,
 1311,
 3,
 2,
 308,
 2261,
 2,
 264,
 168,
 2,
 5750,
 7,
 254,
 14,
 966,
 4,
 76,
 264,
 110,
 75,
 1298,
 511,
 444,
 14,
 1043,
 3,
 39,
 504,
 110,
 35,
 1795,
 1995,
 2636,
 27,
 2,
 7255,
 5,
 64,
 504,
 57,
 2636,
 4,
 1886,
 264,
 50,
 9,
 440,
 6,
 1299,
 5,
 1182,
 893,
 21,
 38,
 6017,
 0,
 4,
 3002,
 8,
 808,
 504,
 110,
 35,
 1066,
 8,
 9,
 221,
 2090,
 4,
 216,
 2698,
 3,
 504,
 205,
 1326,
 85,
 236,
 1284,
 8522,
 8,
 96,
 3,
 91,
 16,
 38,
 1418,
 634,
 23,
 0,
 22,
 2699,
 473,
 46,
 103,
 5606,
 85,
 21,
 1844,
 1199,
 4,
 1886,
 264,
 50,
 1349,
 11,
 0,
 11,
 3,
 3849,
 1864,
 8,
 174,
 264,
 4,
 156,
 36,
 1675,
 63,
 11,
 6018,
 0,
 11,
 3,
 33,
 36,
 0,
 3849,
 15,
 1509,
 0,
 5607,
 3542,
 0,
 21,
 2,
 373,
 6,
 110,
 581,
 647,
 46,
 0,
 9,
 26

In [47]:
# Quickly convert all words to ids

# Break things into words
train_text = [doc['text'].split() for doc in wikitext['train']]
valid_text = [doc['text'].split() for doc in wikitext['validation']]

# Just remove all docs which have no words
train_text = [x for x in train_text if len(x)>0]
valid_text = [x for x in valid_text if len(x)>0]

# Use vocab to turn them into ids
train_text_ids = []
for doc in train_text:
    train_text_ids.append(
        [vocab.get(word, vocab['<unk>']) for word in doc]
    )

valid_text_ids = []
for doc in valid_text:
    valid_text_ids.append(
        [vocab.get(word, vocab['<unk>']) for word in doc]
    )

In [50]:
train_text[1][:10], train_text_ids[1][:10]

(['Senjō',
  'no',
  'Valkyria',
  '3',
  ':',
  '<unk>',
  'Chronicles',
  '(',
  'Japanese',
  ':'],
 [0, 127, 3908, 90, 43, 0, 4419, 23, 752, 43])

In [ ]:
# <pad> <pad> make him an offer he cant refuse <pad> <pad>
# 0. <pad> <pad> him an -> make
# ?. <pad> make an offer -> him
# 1. make him offer he -> an
# 2. him an he cant -> offer
# 3. an offer cant refuse -> he
# P(he | an offer cant refuse)

In [51]:
doc = train_text[0]
_doc = ['<pad>', '<pad>'] + doc + ['<pad>', '<pad>']
i =4
_i = i+2
print(_doc)
_doc[_i-2:_i], _doc[_i+1:_i+3], _i, _i-2, _doc[:10]

['<pad>', '<pad>', '=', 'Valkyria', 'Chronicles', 'III', '=', '<pad>', '<pad>']


(['Chronicles', 'III'],
 ['<pad>', '<pad>'],
 6,
 4,
 ['<pad>',
  '<pad>',
  '=',
  'Valkyria',
  'Chronicles',
  'III',
  '=',
  '<pad>',
  '<pad>'])

In [ ]:
# Making data loaders

# We want to have inputs be [w_-2, w_-1, w_+1, w_+2]. The label for this instance would be w

contexts, targets = [], []
for doc in tqdm(train_text):
    _doc = ['<pad>', '<pad>'] + doc + ['<pad>', '<pad>']
    for i, word in enumerate(doc):
        # hint _i = i +2
        _i = i + 2
        contexts.append(_doc[_i-2:_i]+ _doc[_i+1:_i+3])
        targets.append(_doc[_i])
    break

contexts, targets

In [53]:
# Scale it up for the entire dataset
pad_id = vocab['<pad>']

train_contexts, train_targets = [], []
for doc in tqdm(train_text_ids):
    _doc = [pad_id, pad_id] + doc + [pad_id, pad_id]
    for i, word in enumerate(doc):
        _i = i + 2
        train_contexts.append( _doc[_i-2:_i] + _doc[_i+1:_i+3])
        train_targets.append(_doc[_i])

valid_contexts, valid_targets = [], []
for doc in tqdm(valid_text_ids):
    _doc = [pad_id, pad_id] + doc + [pad_id, pad_id]
    for i, word in enumerate(doc):
        _i = i + 2
        valid_contexts.append( _doc[_i-2:_i] + _doc[_i+1:_i+3])
        valid_targets.append(_doc[_i])


print(len(train_contexts), len(train_targets), len(valid_contexts), len(valid_targets))

100%|██████████| 2461/2461 [00:00<00:00, 9176.53it/s]

2051910 2051910 213886 213886


In [ ]:
)

tensor([  10, 3908, 4419,  ..., 2487, 5367,    4])

In [ ]:
# Throw them into a dataloader
train_contexts = torch.tensor(train_contexts, dtype=torch.long)
train_targets = torch.tensor(train_targets, dtype=torch.long)
valid_contexts = torch.tensor(valid_contexts, dtype=torch.long)
valid_targets = torch.tensor(valid_targets, dtype=torch.long)
print(train_contexts.shape, train_targets.shape)

cbow_train_dataset = TensorDataset(train_contexts, train_targets)
cbow_valid_dataset = TensorDataset(valid_contexts, valid_targets)

train_dataloader = DataLoader(cbow_train_dataset, batch_size=512, shuffle=True)
valid_dataloader = DataLoader(cbow_valid_dataset, batch_size=512, shuffle=True)

In [65]:
## Try it out:
for batch in train_dataloader:
    inputs, targets = batch
    break

inputs, targets

(tensor([[2427,  168, 6856,    4],
         [1793,    5, 1824,    3],
         [ 958, 3064,  434,  757],
         [   3,    2,    6, 1888],
         [ 183, 2439,    9,  146],
         [   2,   82, 1122,   18],
         [   3,  689,   46,  111],
         [ 744,   60,  918,  861],
         [   0,    0,    0,    3],
         [  94, 6640,    2,    0]]),
 tensor([1194,    2,  595, 1139,    7,  452,    3,    2,    3,   63]))

In [ ]:
# p(y=<unk>|x=[2427,  168, 6856,    4])
# p(y=<pad>|x=[2427,  168, 6856,    4])
# p(y=<the>|x=[2427,  168, 6856,    4])
# p(y=<.>|x=[2427,  168, 6856,    4])
# p(y=<,>|x=[2427,  168, 6856,    4])
# p(y=<,>|x=[2427,  168, 6856,    4])

In [ ]:
# torch.randn(10_000, 5)[[2427,  168, 6856,    4]].mean(axis=1)

tensor([0.4858, 0.0220, 0.3414, 0.5851])

![cbow](<../resources/cbow.png>)

# 2. Model

1. We start with the four context words
2. We assign each a vector (4 vectors, n dimensions) using an embedding layer/matrix
3. We average four vectors to create a 'context vector'
4. We pass the 'context vector' to the output layer
5. We get a probability distribution over the vocabulary

In [70]:
# Lets do it without a class now
inputs = torch.randint(1, 10_000, (1, 4))
inputs

tensor([[4306, 4875, 8861,  537]])

In [ ]:
embeddings = torch.nn.Embedding(len(vocab),300)
input_vectors = embeddings(inputs)
input_vectors

tensor([[[ 0.0398, -0.7071,  1.5930,  ...,  0.0726,  0.2574, -0.1602],
         [ 0.0504,  1.8788, -1.1065,  ...,  0.2796, -0.7904,  0.8237],
         [-0.5212,  0.6975, -0.4799,  ...,  1.6603, -0.2432,  0.2063],
         [-0.7192, -0.2286,  0.4933,  ..., -0.3697,  0.4311, -2.1195]]],
       grad_fn=<EmbeddingBackward0>)

In [ ]:
context_vector = input_vectors.mean(dim=1)
context_vector.shape

torch.Size([1, 300])

In [92]:
lin = torch.nn.Linear(300, len(vocab))
logits = lin(context_vector)
logits, logits.shape

(tensor([[-0.0971, -0.2050,  0.6479,  ..., -0.2869, -0.3647,  0.0554]],
        grad_fn=<AddmmBackward0>),
 torch.Size([1, 10001]))

In [ ]:
# torch.softmax(logits, dim=1)

tensor([[8.6886e-05, 7.8000e-05, 1.8302e-04,  ..., 7.1867e-05, 6.6486e-05,
         1.0120e-04]], grad_fn=<SoftmaxBackward0>)

In [ ]:
# Lets make a class out of this
class CBOW(nn.Module):
    def __init__(self, n_words, n_dim):
        super().__init__()
        self.embeddings = torch.nn.Embedding(n_words, n_dim)
        self.lin = torch.nn.Linear(n_dim, n_words)

    def forward(self, inputs):
        input_vectors = self.embeddings(inputs)
        context_vector = input_vectors.mean(dim=1)
        logits = self.lin(context_vector)
        return logits

---------------


# So let's start backproping???

![computetime](https://media.tenor.com/rDKZFPwK-00AAAAC/the-matrix-keanu-reeves.gif "backprop")

In [ ]:
# Do the training
model = CBOW(n_dim=300, n_words=len(vocab))
lfn = torch.nn.CrossEntropyLoss()
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
epochs = 5

In [ ]:

import numpy as np
losses = []
for e in range(epochs):
    epoch_losses = []
    for batch in tqdm(train_dataloader):
        inputs, target = batch

        opt.zero_grad()
        logits = model(inputs)
        loss = lfn(logits, target)
        loss.backward()
        opt.step()

        epoch_losses.append(loss.cpu().detach().item())
    losses.append(epoch_losses)
    tqdm.write(f"Epoch {e}: Loss - {np.mean(epoch_losses)} ")

# Too slow?

Lets use a GPU

In [ ]:
import numpy as np

device = 'mps'
model = model.to(device)

losses = []
for e in range(epochs):
    epoch_losses = []
    for batch in tqdm(train_dataloader):
        inputs, target = batch
        inputs = inputs.to(device)
        target = target.to(device)

        opt.zero_grad()
        logits = model(inputs)
        loss = lfn(logits, target)
        loss.backward()
        opt.step()

        epoch_losses.append(loss.cpu().detach().item())
    losses.append(np.mean(epoch_losses))
    tqdm.write(f"Epoch {e}: Loss - {losses[-1]:.4f} ")

In [106]:
model.to(device)

CBOW(
  (embeddings): Embedding(10001, 300)
  (lin): Linear(in_features=300, out_features=10001, bias=True)
)

In [104]:
a = torch.randn(5,5, device=device)
b = torch.randn(5,5, device=device)
a, b, a*b

(tensor([[-0.9814,  1.7478,  1.4958, -0.8116, -0.2926],
         [ 1.4682, -2.8182, -0.3878, -1.0174, -0.5368],
         [-0.8118, -0.7031, -3.5174, -0.4861, -0.1337],
         [-0.9108,  0.7766,  0.3267,  0.1678, -0.1705],
         [-0.4257,  0.7511,  2.0499,  0.8145,  0.8033]], device='mps:0'),
 tensor([[-0.6281, -0.5797, -1.1954, -0.1166,  1.6352],
         [-0.3133, -1.2807, -2.5721,  1.7529, -0.7976],
         [ 1.6592,  1.8154, -1.1368,  2.0308,  0.9310],
         [-1.0820, -0.6193, -0.0548,  0.7461,  1.3445],
         [-0.5664, -0.7857,  1.2120,  2.2216, -2.3886]], device='mps:0'),
 tensor([[ 0.6164, -1.0132, -1.7881,  0.0946, -0.4785],
         [-0.4599,  3.6092,  0.9974, -1.7835,  0.4281],
         [-1.3468, -1.2764,  3.9984, -0.9871, -0.1245],
         [ 0.9855, -0.4809, -0.0179,  0.1252, -0.2292],
         [ 0.2411, -0.5902,  2.4845,  1.8095, -1.9188]], device='mps:0'))

# <font color="red">Problems! </font>: Inefficient

For each word pair, we compute a distribution over the enitre vocabulary.
Why? To normalize the scores.

###### Recall: 

**score**: $f(u.v)$ or $(u^T v)$. Our $f(.)$ was $\text{exp}(.)$


**normalization**: $\sum_{i=0}^{|\text{vocab}|} f(u.v_i)$ <- **nicht gut!**


# Further Reading

A great overview of this entire thing - [Blogpost](https://mccormickml.com/2016/04/19/word2vec-tutorial-the-skip-gram-model/)


Another implementation of the entire thing - [Github](https://github.com/lukysummer/SkipGram_with_NegativeSampling_Pytorch/blob/master/SkipGram_NegativeSampling.ipynb)

Skip-Gram embeddings with negative embeddings is implicit factorization of the co-occurance matrix - [Paper](https://papers.nips.cc/paper/2014/file/feab05aa91085b7a8012516bc3533958-Paper.pdf)

On Biases in Word Embeddings, and ways to counteract them (ony gender bias targeted in this paper) - [Paper](https://arxiv.org/pdf/1607.06520.pdf)

WEAT Test - [Paper](https://arxiv.org/pdf/1608.07187.pdf)
